# Generación solar fotovoltaica en España: forecasting

Estimamos la producción fotovoltaica diaria de la península en GWh a partir del historial, el calendario y, opcionalmente, la radiación solar diaria de Madrid. La producción se obtiene de Red Eléctrica y la radiación de Open-Meteo.

**En sencillo:** la generación solar depende de patrones pasados y de cuánta luz solar haya. Comparamos el modelo con y sin radiación y probamos con las fechas más recientes, sin mezclar la serie.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ml_al_alfa.datasets import FORECAST_CONFIG, load_forecast_frame
from ml_al_alfa.forecasting import forecast_to_date, train_forecast_and_save
from ml_al_alfa.train import ARTIFACTS_DIR, RAW_DIR

sns.set_theme(style="whitegrid")
DATASET = "solar"
CONFIG = FORECAST_CONFIG[DATASET]
TARGET = CONFIG["target"]
WEATHER = CONFIG["exogenous"][0]

## 1. Obtener y revisar los datos

Si todavía no existe el archivo preparado, el proyecto descarga la generación diaria de Red Eléctrica y el historial meteorológico de Madrid desde Open-Meteo y guarda la combinación en `data/raw/`.

In [ ]:
series = load_forecast_frame(DATASET, RAW_DIR)
series["date"] = pd.to_datetime(series["date"])
series = series.sort_values("date").reset_index(drop=True)
print(f"Filas: {len(series):,}; fechas: {series['date'].min().date()} – {series['date'].max().date()}")
print(f"Valores ausentes: {series[[TARGET, WEATHER]].isna().sum().sum()}")
display(series.head())
display(series[[TARGET, WEATHER]].describe().T)

## 2. Explorar producción y radiación

La producción cambia según estación y nubosidad. La radiación meteorológica corresponde a Madrid, no es una medición de todas las plantas solares de España.

In [ ]:
series["día de la semana"] = series["date"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
axes[0].plot(series["date"], series[TARGET], linewidth=0.8, color="darkorange")
axes[0].set(title="Generación solar fotovoltaica diaria", xlabel="Fecha", ylabel="GWh")
axes[1].plot(series["date"], series[WEATHER], linewidth=0.8, color="goldenrod")
axes[1].set(title="Radiación solar diaria en Madrid", xlabel="Fecha", ylabel="MJ/m²")
sns.boxplot(data=series, x="día de la semana", y=TARGET, order=weekday_order, ax=axes[2])
axes[2].tick_params(axis="x", rotation=25)
axes[2].set(title="Producción por día de la semana", xlabel="Día", ylabel="GWh")
plt.tight_layout()
plt.show()

sns.scatterplot(data=series, x=WEATHER, y=TARGET, alpha=0.4, color="darkorange")
plt.title("Producción fotovoltaica y radiación de Madrid")
plt.xlabel("Radiación (MJ/m²)")
plt.ylabel("Generación (GWh)")
plt.show()

## 3. Entrenar sin barajar las fechas

El baseline usa la generación del mismo día de la semana anterior. Comparamos regresión lineal y Random Forest con calendario e historial, y volvemos a compararlos con radiación observada. Para pronosticar fechas futuras, introduce una estimación de radiación; el valor histórico del test no se conoce de antemano.

In [ ]:
bundle = train_forecast_and_save(
    series[["date", TARGET, WEATHER]],
    dataset_name=DATASET,
    output_path=ARTIFACTS_DIR / "solar_model.joblib",
)
metrics = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(metrics.rename(columns={"mae": "MAE (GWh/día, menor es mejor)", "r2": "R²"}))
print(f"Inicio del test cronológico: {bundle['forecast']['split_date']}")
print(f"Mejor variante: {bundle['forecast']['best_variant']}")
print(f"Mejor método: {bundle['model_name']}")
print(f"Modelo para Streamlit: {ARTIFACTS_DIR / 'solar_model.joblib'}")

## 4. Revisar el test

Comparamos generación real y estimada en las fechas reservadas para medir el error en la misma unidad del problema.

In [ ]:
evaluation = pd.DataFrame(bundle["forecast"]["evaluation"])
evaluation["date"] = pd.to_datetime(evaluation["dates"])
display(evaluation[["date", "actual", "predicted"]].tail(10))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(evaluation["date"], evaluation["actual"], label="Real", linewidth=1)
ax.plot(evaluation["date"], evaluation["predicted"], label="Pronóstico", linewidth=1)
ax.set(title="Pronósticos frente a generación solar real", xlabel="Fecha", ylabel="Generación (GWh)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Pronosticar una fecha futura

Para el ejemplo usamos la radiación mediana histórica como supuesto; no es un pronóstico meteorológico. Para planificar de verdad, sustituye el valor por una previsión meteorológica para esas fechas.

In [ ]:
radiation_assumption = {WEATHER: float(series[WEATHER].median())}
future = forecast_to_date(
    bundle,
    variant_label=bundle["forecast"]["best_variant"],
    days_ahead=7,
    exogenous_values=radiation_assumption,
)
future_frame = pd.DataFrame(future, columns=["date", "estimated_solar_gwh"])
display(future_frame)
print(f"Supuesto diario de radiación: {radiation_assumption[WEATHER]:.1f} MJ/m²")
print("En Streamlit introduce la radiación prevista para el horizonte elegido.")